In [8]:
import pandas as pd
engine_df = pd.read_excel("2023_Price_Adjustment_New_and_ReCon_NESEA_Full details w engine.xlsx", sheet_name="NESEA - New - 2023")
engine2_df = pd.read_excel("engine modelxlsx.xlsx", sheet_name="Sheet2")

In [9]:
print(engine_df.columns)

Index(['PART_NO', 'PRINT_PART_NO', 'PN AL78', 'PART_DESC', 'QTY_IN_CARTON',
       'ENGINE_APPL_CODE', 'CURRENT_PVC_CODE', 'FUTURE_PVC_CODE',
       'CHANGE_REASON', 'ECC', 'WPT', 'Engine Model', 'Displacement', 'ENGINE',
       'Unnamed: 14'],
      dtype='object')


In [13]:
print(engine_df)

            PART_NO PRINT_PART_NO PN AL78           PART_DESC  QTY_IN_CARTON  \
0         000362600          3626    3626            FLYWHEEL            1.0   
1         000485100          4851    4851  GEAR,FLYWHEEL RING            1.0   
2         000504500          5045    5045  GEAR,FLYWHEEL RING            1.0   
3         000508300          5083    5083      GASKET,OIL PAN            1.0   
4         000556600          5566    5566  GEAR,FLYWHEEL RING            1.0   
...             ...           ...     ...                 ...            ...   
109332  OHK945700 D      OHK9457D     NaN       KIT, OVERHAUL            1.0   
109333  OHK945800 D      OHK9458D     NaN       KIT, OVERHAUL            1.0   
109334  OHK945900 D      OHK9459D     NaN       KIT, OVERHAUL            1.0   
109335  OHK946300 D      OHK9463D     NaN       KIT, OVERHAUL            1.0   
109336  OHK946500 D      OHK9465D     NaN      KIT,  OVERHAUL            1.0   

                  ENGINE_APPL_CODE  CUR

In [10]:
print(engine2_df.columns)

Index(['PN', 'Long\nPN', 'Description', 'Engine\nWW32', 'Unnamed: 4'], dtype='object')


In [14]:
print(engine2_df)

                PN     Long\nPN             Description  \
0           106069    010606900  SCREW,HEXAGON HEAD CAP   
1           107460    010746000                    CLIP   
2           107695    010769500                PIN,ROLL   
3           107981    010798100              CAP,FILLER   
4           108722    010872200                    CLIP   
...            ...          ...                     ...   
7477       S   608    S00060800             WASHER,LOCK   
7478       S   962    S00096200               PLUG,PIPE   
7479  S   962    E  S00096200 E               DRAINCOCK   
7480  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER   
7481       S  2268    S00226800          FITTING,GREASE   

                                         Engine\nWW32 Unnamed: 4  
0                              K38, QSK38, K50, QSK50        NaN  
1            QSK19, QSK23, QSK50, QST30, QSK45, QSK60        NaN  
2                          QSK19, QSK23, QSK50, QSK60        NaN  
3                      

In [11]:
# === SELECT & RENAME COLUMNS FROM ENGINE_DF ===
engine_df_sel = engine_df[[
    "PRINT_PART_NO",
    "PART_NO",
    "PART_DESC",
    "ECC",
    "WPT",
    "Engine Model",
    "Displacement",
    "ENGINE"
]].rename(columns={
    "PRINT_PART_NO": "PN AL78",
    "PART_NO": "Long PN",
    "PART_DESC": "Description"
})

# === SELECT & RENAME COLUMNS FROM ENGINE2_DF ===
engine2_df_sel = engine2_df[[
    "PN",
    "Long\nPN",
    "Description",
    "Engine\nWW32"
]].rename(columns={
    "PN": "PN AL78",
    "Long\nPN": "Long PN_2",
    "Description": "Description_2",
    "Engine\nWW32": "Engine_2"
})

# === MERGE (NO DUPLICATE PN AL78) ===
final_df = engine_df_sel.merge(
    engine2_df_sel,
    on="PN AL78",
    how="left"
)

# === PRIORITIZE ENGINE_DF VALUES, FILL FROM ENGINE2_DF IF NULL ===
final_df["Long PN"] = final_df["Long PN"].fillna(final_df["Long PN_2"])
final_df["Description"] = final_df["Description"].fillna(final_df["Description_2"])
final_df["ENGINE"] = final_df["ENGINE"].fillna(final_df["Engine_2"])

# === DROP EXTRA COLUMNS ===
final_df = final_df.drop(columns=["Long PN_2", "Description_2", "Engine_2"])

# === FINAL COLUMN ORDER ===
final_df = final_df[[
    "PN AL78",
    "Long PN",
    "Description",
    "ECC",
    "WPT",
    "Engine Model",
    "Displacement",
    "ENGINE"
]]

print(final_df.head())


   PN AL78    Long PN         Description                    ECC  WPT  \
0     3626  000362600            FLYWHEEL  Flywheel {Analytical}   HD   
1     4851  000485100  GEAR,FLYWHEEL RING      Gear {Analytical}   HD   
2     5045  000504500  GEAR,FLYWHEEL RING      Gear {Analytical}  HHP   
3     5083  000508300      GASKET,OIL PAN    Gasket {Analytical}   HD   
4     5566  000556600  GEAR,FLYWHEEL RING      Gear {Analytical}   HD   

  Engine Model Displacement ENGINE  
0          855           14  NT855  
1          855           14  NT855  
2            V           28  VTA28  
3          855           14  NT855  
4          855           14  NT855  


In [12]:
print(final_df)

         PN AL78      Long PN         Description                    ECC  WPT  \
0           3626    000362600            FLYWHEEL  Flywheel {Analytical}   HD   
1           4851    000485100  GEAR,FLYWHEEL RING      Gear {Analytical}   HD   
2           5045    000504500  GEAR,FLYWHEEL RING      Gear {Analytical}  HHP   
3           5083    000508300      GASKET,OIL PAN    Gasket {Analytical}   HD   
4           5566    000556600  GEAR,FLYWHEEL RING      Gear {Analytical}   HD   
...          ...          ...                 ...                    ...  ...   
114443  OHK9457D  OHK945700 D       KIT, OVERHAUL  Overhaul Kit {Active}   HD   
114444  OHK9458D  OHK945800 D       KIT, OVERHAUL  Overhaul Kit {Active}   HD   
114445  OHK9459D  OHK945900 D       KIT, OVERHAUL  Overhaul Kit {Active}   HD   
114446  OHK9463D  OHK946300 D       KIT, OVERHAUL  Overhaul Kit {Active}   HD   
114447  OHK9465D  OHK946500 D      KIT,  OVERHAUL  Overhaul Kit {Active}   HD   

       Engine Model Displac

In [15]:
# =========================================================
# EXPORT
# =========================================================
final_df.to_excel("Engine model combined.xlsx", index=False)